# Graph Neural Networks (GNN): Message Passing, GCN, GraphSAGE & GAT

This interactive lab covers:
1. **Graph Message Passing Mechanics**: Structural graph representations ($A, D, X$).
2. **Graph Convolutional Networks (GCN)**: Symmetric Laplacian normalization and spectral filtering.
3. **GraphSAGE**: Inductive neighborhood aggregation.
4. **Graph Attention Networks (GAT)**: Dynamic edge-level self-attention weighting.
5. **Semi-Supervised Node Classification**: Training a 2-layer GCN on community graphs.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import torch
import torch.nn as nn
import torch.optim as optim
from gnn_engine import GCNLayer, GraphSAGELayer, GATLayer, GCNNodeClassifier

print('GNN Engine modules loaded successfully!')

## 1. Graph Representation & Symmetric Laplacian Normalization

We set up a 5-node graph with two connected communities and compute the renormalized adjacency matrix $\hat{A} = \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$.

In [ ]:
# Adjacency matrix for 5-node graph: (0-1-2) community, (3-4) community, connected by edge (2-3)
A = torch.tensor([
    [0., 1., 1., 0., 0.],
    [1., 0., 1., 0., 0.],
    [1., 1., 0., 1., 0.],
    [0., 0., 1., 0., 1.],
    [0., 0., 0., 1., 0.]
])

A_norm = GCNLayer.normalize_adjacency(A)
print('Symmetrically Normalized Adjacency Matrix A_norm:')
print(A_norm.numpy().round(3))
assert torch.allclose(A_norm, A_norm.T), 'Normalized adjacency must be symmetric!'

## 2. Comparing Message Passing Architectures (GCN vs. GraphSAGE vs. GAT)

We pass node features $X \in \mathbb{R}^{5 \times 4}$ through each layer and inspect outputs and attention weights.

In [ ]:
torch.manual_seed(42)
X = torch.randn(5, 4)

gcn = GCNLayer(in_features=4, out_features=2)
sage = GraphSAGELayer(in_features=4, out_features=2)
gat = GATLayer(in_features=4, out_features=2)

out_gcn = gcn(X, A_norm)
out_sage = sage(X, A)
out_gat, attn_weights = gat(X, A + torch.eye(5))

print(f'GCN Output Shape:       {out_gcn.shape}')
print(f'GraphSAGE Output Shape: {out_sage.shape}')
print(f'GAT Output Shape:       {out_gat.shape}')
print('\nGAT Attention Weights for Node 2 (connects both communities):')
print(attn_weights[2].detach().numpy().round(3))

## 3. Semi-Supervised Node Classification with 2-Layer GCN

We train our `GCNNodeClassifier` to predict community membership (Class 0: Nodes 0, 1, 2; Class 1: Nodes 3, 4) using cross-entropy loss.

In [ ]:
model = GCNNodeClassifier(in_features=4, hidden_dim=8, num_classes=2, dropout=0.0)
optimizer = optim.Adam(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

labels = torch.tensor([0, 0, 0, 1, 1])

for epoch in range(50):
    optimizer.zero_grad()
    logits = model(X, A_norm)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()

preds = torch.argmax(logits, dim=1)
acc = (preds == labels).float().mean().item()
print(f'Trained GCN Final Loss: {loss.item():.4f}')
print(f'Predicted Node Labels: {preds.tolist()} | Ground Truth: {labels.tolist()}')
print(f'Node Classification Accuracy: {acc * 100:.0f}%')
assert acc == 1.0, 'GCN model did not achieve 100% accuracy on toy community graph!'